# Colditz (1994): Meta-Analysis

When there are multiple studies focused on the same question, one might be interested in meta-analysis, whereby the results of those multiple studies are integrated together to boost precision. To illustrate how `delicatessen` can be used to conduct meta-analyses, we re-examine data from Colditz et al. (1994), which used meta-analysis to estimate the efficacy of BCG vaccination for prevention of tuberculosis (TB). The corresponding data can be obtained from the R package `metafor`.

## Setup

In [1]:
import numpy as np
import scipy as sp
import pandas as pd
import delicatessen as deli
from delicatessen import MEstimator
from delicatessen.estimating_equations import (ee_mean, ee_meta_random,
                                               ee_regression, ee_meta_regression)

print("Versions")
print("NumPy:        ", np.__version__)
print("SciPy:        ", sp.__version__)
print("Pandas:       ", pd.__version__)
print("Delicatessen: ", deli.__version__)

Versions
NumPy:         2.3.5
SciPy:         1.16.3
Pandas:        2.3.3
Delicatessen:  4.2


In [2]:
# Loading data from metafor
d = pd.read_csv("data/colditz1994.csv")

The provided data describes the events per arm. We talk this raw data and convert it into summary measures instead. Explicitly, we manually estimate the risk differences and risk ratios. In general, this is what would be available to conduct a meta-analysis. 

In [3]:
d['n'] = d['tpos'] + d['tneg'] + d['cpos'] + d['cneg']
d['r1'] = d['tpos'] / (d['tpos'] + d['tneg'])
d['r0'] = d['cpos'] / (d['cpos'] + d['cneg'])
d['rd'] = d['r1'] - d['r0']
d['var_rd'] = ((d['r1'] * (1 - d['r1'])) / (d['tpos'] + d['tneg'])
               + (d['r0'] * (1 - d['r0'])) / (d['cpos'] + d['cneg']))
d['rr'] = d['r1'] / d['r0']
d['var_lrr'] = ((1 / d['tpos']) - (1 / (d['tpos'] + d['tneg']))
                + (1 / d['cpos']) - (1 / (d['cpos'] + d['cneg'])))
d['c'] = 1

## Common-Effect Meta-Analysis

In the first case, we can simply take a weighted average of the point estimates. Here, we use the inverse of the estimated variance as the weight. This means that studies that are more precise (i.e., smaller variance), have a larger contribution to the overall estimate relative to studies that are imprecise. In meta-analysis, this approach is commonly referred to as common-effect or equal-effect models.

As this is simply a weighted average of the point estimates, we can use the existing `ee_mean` function. Here, the risk difference is considered first

In [4]:
def psi_rd(theta):
    return ee_mean(theta=theta, y=d['rd'], weights=1/d['var_rd'])

In [5]:
estr = MEstimator(psi_rd, init=[0., ], finite_correction='hc1')
estr.estimate()
estr.print_results(decimals=3)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:          13 | No. Parameters:              1
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:       hc1 | Distribution:           t-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
  -0.001    0.001   -1.405   -0.002    0.001    0.185    2.431 


On the risk difference scale, we see little change by vaccination. In general, the risk of TB was low in the data so this is not surprising.

We can also repeat this process for the risk ratio. However, we can no longer simply take the mean of the risk ratios for the meta-analysis. For the average, we want the point estimates to combine linearly. Therefore, we instead meta-analyze the *log risk ratios* and the variance of the *log risk ratios*. The following code applies this process in `delicatessen`

In [6]:
def psi_rr(theta):
    return ee_mean(theta=theta, y=np.log(d['rr']), weights=1/d['var_lrr'])

In [7]:
estr = MEstimator(psi_rr, init=[0., ], finite_correction='hc1')
estr.estimate()
estr.print_results(decimals=3)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:          13 | No. Parameters:              1
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:       hc1 | Distribution:           t-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
  -0.430    0.229   -1.878   -0.930    0.069    0.085    3.558 


which corresponds to a risk ratio of 0.65. This example highlights how the different scales interact with the absolute risk in different ways.

Here, we also apply the HC1 finite-sample correction for the variance since we have relatively few observations.

## Random-Effect Meta-Analysis

The previous common-effect approach assumes that all studies include in the meta-analysis share a risk ratio. This is unlikely to be the case. Instead, we can use a random-effects model which assumes that the included studies are drawn from a distribution of true effects. In other words, heterogeneity in the risk ratio between studies is allowed.

This approach involves the estimation of an additional parameter corresponding to the heterogeneity between studies. Random-effect meta-analysis is enabled through the `ee_meta_random` estimating functions. Implicitly, the method being used to combine studies in the random-effects model is the Paule-Mandel method, which is equivalent to Empirical Bayes (Viechtbauer et al. 2015). Here, we apply this approach to meta-analyze the risk ratios

In [8]:
def psi_mr(theta):
    return ee_meta_random(theta=theta, point_est=np.log(d['rr']), var_est=d['var_lrr'])

In [9]:
estr = MEstimator(psi_mr, init=[np.mean(np.log(d['rr'])), 0.], finite_correction='hc1')
estr.estimate()
estr.print_results(decimals=3)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:          13 | No. Parameters:              2
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:       hc1 | Distribution:           t-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
  -0.715    0.188   -3.801   -1.129   -0.301    0.003    8.411 
  -1.145    0.270   -4.244   -1.740   -0.551    0.001    9.501 


Here, we see a stronger risk ratio (`exp(-0.715)=0.50`) that seems substantially different from the null (based on the S-value). Here also see that the second parameter, which is the log-transformed heterogeneity parameter is far from zero. Therefore, there is some underlying heterogeneity between these studies.

One caution is that the random-effect meta-analysis can perform poorly when there is few studies being meta-analyzed. Therefore, this approach is not to be recommended when there are fewer than 10 studies.

## Random-Effect Meta-Regression

Beyond simply summarizing the point estimates, we can also model how the point estimates vary by characteristics of the studies. For example, we can model how the estimated risk ratios change as a function of the latitude of the study location.

To simplify interpretation and improve computational performance, we take several initial steps. First, the latitude of the study locations are centered, so that the intercept corresponds to the mean latitude (rather than a latitude of zero). The second part is that we first fit a common-effect meta-regression model. The parameters of that model are used as starting values for the random-effect meta-regression model. The following code applies these two steps

In [10]:
d['ablat_c'] = d['ablat'] - np.mean(d['ablat'])

In [11]:
# Finding good initial values by fitting a common-effect meta-regression
def psi_reg(theta):
    return ee_regression(theta=theta, y=np.log(d['rr']),
                         X=d[['c', 'ablat_c']], model='linear',
                         weights=1/d['var_lrr'])

estr = MEstimator(psi_reg, init=[-0.1, -0.03])
estr.estimate()
init_beta = list(estr.theta)

Having done this background preparation, we can now estimate the parameters of the random-effects meta-regression model

In [12]:
def psi_mreg(theta):
    return ee_meta_regression(theta=theta, point_est=np.log(d['rr']),
                              var_est=d['var_lrr'],
                              X=d[['c', 'ablat_c']])

In [13]:
estr = MEstimator(psi_mreg, init=init_beta + [0.1, ])
estr.estimate()
estr.print_results(decimals=4)

              Estimation Method: M-estimator
--------------------------------------------------------------
No. Observations:          13 | No. Parameters:              3
Solving algorithm:         lm | Max Iterations:           5000
Solving tolerance:      1e-09 | Allow P-Inverse:             1
Derivative Method:     approx | Deriv Approx:            1e-09
Small N Correction:      None | Distribution:           Z-stat
   Theta   StdErr  Z-score      LCL      UCL  P-value  S-value 
--------------------------------------------------------------
 -0.7339   0.1088  -6.7472  -0.9471  -0.5207   0.0000  35.9495 
 -0.0286   0.0059  -4.8184  -0.0402  -0.0169   0.0000  19.3985 
 -1.9510   0.8860  -2.2019  -3.6876  -0.2144   0.0277   5.1755 


In this model, we can see the risk ratio for the mean latitude (first parameter), how the risk ratio varies by a one degree change in latitude (second parameter), and the estimated heterogeneity. One will notice here that the point estimate for heterogeneity decreased somewhat. This is indicative of latitude playing a role in explaining the heterogeneity of results between studies.

The advantage of `delicatessen` for meta-analysis is that the estimating equations automatically accomodate the dependence of the summary risk ratios on the heterogeneity parameter. Other methods for meta-analysis and meta-regression do not always appropriately incorporate this variability into the parameters of interest. As described for the DerSimonian–Laird estimator, the heterogeneity parameter is assumed to be known, which can then under-estimate the variance (Cornell et al. 2014). `delicatessen` solves this issue through the use of the empirical sandwich variance estimator

## References

Colditz GA, Brewer TF, Berkey CS, Wilson ME, Burdick E, Fineberg HV, & Mosteller F. (1994). Efficacy of BCG vaccine in the prevention of tuberculosis: meta-analysis of the published literature. *JAMA*, 271(9), 698-702.

Cornell JE, Mulrow CD, Localio R, Stack CB, Meibohm AR, Guallar E, & Goodman SN. (2014). Random-effects meta-analysis of inconsistent effects: a time for change. *Annals of Internal Medicine*, 160(4), 267-270.

Paule RC, & Mandel J. (1982). Consensus values and weighting factors. *Journal of research of the National Bureau of Standards*, 87(5), 377.

Viechtbauer W, López-López JA, Sánchez-Meca J, & Marín-Martínez F. (2015). A comparison of procedures to test for moderators in mixed-effects meta-regression models. *American Psychological Association*, 20(3), 360.

Veroniki AA, Jackson D, Viechtbauer W, et al. (2016). Methods to estimate the between‐study variance and its uncertainty in meta‐analysis. *Research Synthesis Methods*, 7(1), 55-79.